#  Machine Learning para prever Resistência a Antibióticos
## Modelagem

In [2]:
# Imports
import sklearn
import xgboost
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
from sklearn import metrics
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv("dataset_final.csv", na_values = "NaN")

### Seleção dos Dados Usados na Modelagem

In [4]:
# Extrair a lista de medicamentos
lista_medicamentos = df.iloc[:,1:13].columns

In [5]:
# Define a função que cria um DataFrame específico para um medicamento
def cria_df(medicamento):
    
    # Cria uma lista de dataframes contendo a coluna "Isolate" e a coluna do medicamento especificado, 
    # e as colunas a partir da 13ª
    df_list = [df[["Isolate", medicamento]], df.iloc[:, 13:]]
    
    # Concatena os DataFrames da lista ao longo do eixo das colunas (axis=1)
    df_medicamento = pd.concat(df_list, axis = 1)
    
    # Remove as linhas com valores ausentes (NaN)
    df_medicamento = df_medicamento.dropna()
    
    # Retorna o DataFrame resultante
    return df_medicamento

In [6]:
# Cria o dataframe para um medicamento
AMP_df = cria_df("AMP")

In [9]:
# Amostra
print(AMP_df.sample(3))

          Isolate AMP  Year_1970.0  Year_1977.0  Year_1994.0  Year_1997.0  \
13     11657_5#22   S        False        False        False        False   
581    11658_6#24   R        False        False        False        False   
1898  24742_1#365   R        False        False        False        False   

      Year_1998.0  Year_1999.0  Year_2001.0  Year_2002.0  ...  cutoff_25459  \
13          False        False        False        False  ...             0   
581         False        False        False        False  ...             0   
1898        False        False        False        False  ...             0   

      cutoff_25654  cutoff_25772  cutoff_25979  cutoff_26792  cutoff_27119  \
13               0             0             0             0             0   
581              0             0             0             0             0   
1898             0             0             0             0             0   

      cutoff_27236  cutoff_27248  cutoff_27690  cutoff_45092 

In [12]:
# Define a função que divide o DataFrame em conjuntos de treino e teste
def split(df_medicamento, medicamento):
    
    # Inicializa um dicionário para armazenar os conjuntos de treino e teste
    dict_treino_teste = {}
    
    # Extrai as labels (rótulos) do DataFrame, que correspondem ao medicamento especificado
    labels = df_medicamento[medicamento]
    
    # Extrai as features (características), removendo a coluna do medicamento do DataFrame
    features = df_medicamento.drop(columns = [medicamento])
    
    # Divide as features e labels em conjuntos de treino e teste, com 33% dos dados para teste 
    # e uma semente aleatória para reprodutibilidade
    features_train, features_test, labels_train, labels_test = train_test_split(features, 
                                                                                labels, 
                                                                                test_size = 0.33, 
                                                                                random_state = 42)

    # Armazena as labels de treino no dicionário
    dict_treino_teste['labels_train'] = labels_train
    
    # Armazena as features de treino no dicionário
    dict_treino_teste['features_train'] = features_train
    
    # Armazena as labels de teste no dicionário
    dict_treino_teste['labels_test'] = labels_test
    
    # Armazena as features de teste no dicionário
    dict_treino_teste['features_test'] = features_test

    # Retorna o dicionário contendo os conjuntos de treino e teste
    return dict_treino_teste

In [13]:
AMP_dict_treino_teste = split(AMP_df, "AMP") 

In [17]:
print("Medicamento: AMP\n")
for k, df in AMP_dict_treino_teste.items():
    print(k, df.shape)

Medicamento: AMP

labels_train (563,)
features_train (563, 18292)
labels_test (278,)
features_test (278, 18292)


## Estratégia de Engenharia de Atributos - Criando Diferentes Combinações de Atributos

Criaremos a função combina_dados, que nos permitirá escolher quais são as combinações específicas de atributos que queremos usar para treinar nosso modelo. Combinações que serão criadas:

- "G" (prescrição genética)
- "S" (estrutura populacional)
- "GY" (prescrição genética e ano)
- "GS" (prescrição genética e estrutura populacional)

Isso ajuda a reduzir a dimensionalidade dos dados, ao mesmo tempo que mantém informação relevante para treinar o modelo.

In [ ]:
# Criamos uma lista de combinações dos atributos que desejamos de testar com nosso modelo
combo_list = ['G', 'S', 'GY', 'GS'] 

In [18]:
# Define a função que combina dados com base em diferentes combinações de atributos
def combina_dados(features_df, medicamento, combo):

    # Filtra as colunas que começam com "Year"
    year_filter = [col for col in features_df if col.startswith("Year")]
    
    # Seleciona as colunas filtradas relacionadas a anos
    year_feat = features_df[year_filter]

    # Filtra as colunas que começam com "cutoff"
    pop_str_filter = [col for col in features_df if col.startswith("cutoff")]
    
    # Seleciona as colunas filtradas relacionadas à estrutura populacional
    pop_struc_feat = features_df[pop_str_filter]

    # Filtra as colunas que não estão relacionadas à estrutura populacional, não são anos e não são "Isolate"
    gene_presc_filter = [col for col in features_df.columns if col not in pop_str_filter and col not in year_filter and col != "Isolate"]
    
    # Seleciona as colunas filtradas relacionadas à prescrição genética
    gene_presc_feat = features_df[gene_presc_filter]  

    # Se a combinação escolhida for "G" (prescrição genética)
    if combo == 'G':
        
        # Cria uma lista de DataFrames contendo a coluna "Isolate" e as características genéticas
        df_list = [features_df['Isolate'], gene_presc_feat]
        
        # Concatena os DataFrames da lista ao longo do eixo das colunas (axis=1)
        G_feat_df = pd.concat(df_list, axis = 1)
        
        # Remove a coluna "Isolate"
        G_feat_df = G_feat_df.drop(columns = ['Isolate'])
        
        # Retorna o DataFrame resultante
        return G_feat_df

    # Se a combinação escolhida for "S" (estrutura populacional)
    if combo == 'S':
        
        # Cria uma lista de DataFrames contendo a coluna "Isolate" e as características da estrutura populacional
        df_list = [features_df['Isolate'], pop_struc_feat]
        
        # Concatena os DataFrames da lista ao longo do eixo das colunas (axis=1)
        S_feat_df = pd.concat(df_list, axis = 1)
        
        # Remove a coluna "Isolate"
        S_feat_df = S_feat_df.drop(columns = ['Isolate'])
        
        # Retorna o DataFrame resultante
        return S_feat_df

    # Se a combinação escolhida for "GY" (prescrição genética e ano)
    if combo == 'GY':
        
        # Cria uma lista de DataFrames contendo a coluna "Isolate", as características genéticas e os anos
        df_list = [features_df['Isolate'], gene_presc_feat, year_feat]
        
        # Concatena os DataFrames da lista ao longo do eixo das colunas (axis=1)
        GY_feat_df = pd.concat(df_list, axis = 1)
        
        # Remove a coluna "Isolate"
        GY_feat_df = GY_feat_df.drop(columns = ['Isolate'])
        
        # Retorna o DataFrame resultante
        return GY_feat_df

    # Se a combinação escolhida for "GS" (prescrição genética e estrutura populacional)
    if combo == "GS":
        
        # Cria uma lista de DataFrames contendo a coluna "Isolate", as características genéticas 
        # e da estrutura populacional
        df_list = [features_df['Isolate'], gene_presc_feat, pop_struc_feat]
        
        # Concatena os DataFrames da lista ao longo do eixo das colunas (axis=1)
        GS_feat_df = pd.concat(df_list, axis = 1)
        
        # Remove a coluna "Isolate"
        GS_feat_df = GS_feat_df.drop(columns = ['Isolate'])
        
        # Retorna o DataFrame resultante
        return GS_feat_df

In [20]:
# Aplica a função
AMP_GS_train_df = combina_dados(AMP_dict_treino_teste['features_train'], "AMP", "GS")

In [21]:
# Shape
AMP_GS_train_df.shape

(563, 18269)

In [22]:
# Colunas
AMP_GS_train_df.columns

Index(['yeiU', 'yhhS', 'ybaE', 'eutR', 'ibrB', 'ytfP', 'aslB', 'narQ', 'tolR',
       'galM',
       ...
       'cutoff_25459', 'cutoff_25654', 'cutoff_25772', 'cutoff_25979',
       'cutoff_26792', 'cutoff_27119', 'cutoff_27236', 'cutoff_27248',
       'cutoff_27690', 'cutoff_45092'],
      dtype='object', length=18269)